# 04 — Statistical Analysis

**Dataset:** `data/processed/airbnb_nyc_cleaned.csv`  
**Goal:** Test the hypotheses raised in `03_eda.ipynb` with rigorous statistical methods and quantify the relationships between price, location, room type, and host behaviour.

**Every test reports: test statistic, p-value, effect size, and a plain-language business interpretation.**

| Section | Method |
|---|---|
| 4.1 | Setup |
| 4.2 | Descriptive stats & distribution checks (normality) |
| 4.3 | H1: Manhattan vs Brooklyn price — Welch's t-test + Cohen's d |
| 4.4 | H2: Room-type price differences — Kruskal-Wallis + post-hoc Mann-Whitney |
| 4.5 | H3: Multi-lister revenue — Mann-Whitney U + rank-biserial r |
| 4.6 | H4: Borough price ranking — Kruskal-Wallis + Dunn's test |
| 4.7 | H5: OLS regression — log_price ~ room_type + borough + availability + host_listing_count |
| 4.8 | Correlation analysis — Pearson + Spearman |
| 4.9 | Segmentation — revenue and occupancy by host type × borough |
| 4.10 | Statistical Summary |

---
## 4.1 Setup

In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.stats as stats
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams.update({'figure.dpi': 120, 'axes.titleweight': 'bold'})

PROJECT_ROOT = (
    Path.cwd().resolve().parent
    if Path.cwd().resolve().name == 'notebooks'
    else Path.cwd().resolve()
)
DATA_PATH = PROJECT_ROOT / 'data' / 'processed' / 'airbnb_nyc_cleaned.csv'

df = pd.read_csv(DATA_PATH, low_memory=False)
df['last_review'] = pd.to_datetime(df['last_review'], errors='coerce')
df['price_tier']  = pd.Categorical(df['price_tier'],
                                    categories=['Budget','Mid-Range','Premium','Luxury'],
                                    ordered=True)

ALPHA = 0.05


In [ ]:
def report_test(name, stat, p, effect_name="", effect=None, interpretation=""):
    """Standardised test result reporter."""
    sig = "SIGNIFICANT ✓" if p < 0.05 else "NOT significant ✗"
    print(f"{'-'*50}")
    print(f"TEST: {name}")
    print(f"Result: {sig} (stat={stat:.4f}, p={p:.4e})")
    if effect is not None:
        print(f"Effect: {effect_name} = {effect:.4f}")
    if interpretation:
        print(f"Insight: {interpretation}")
    print(f"{'-'*50}
")


def cohens_d(a, b):
    """Cohen's d effect size for two independent samples."""
    na, nb = len(a), len(b)
    pooled_std = np.sqrt(((na - 1) * np.std(a, ddof=1)**2 +
                          (nb - 1) * np.std(b, ddof=1)**2) / (na + nb - 2))
    return (np.mean(a) - np.mean(b)) / pooled_std


def rank_biserial_r(u_stat, n1, n2):
    """Rank-biserial correlation (effect size for Mann-Whitney U)."""
    return 1 - (2 * u_stat) / (n1 * n2)


---
## 4.2 Distribution Checks — Normality

**Why this matters:** Parametric tests (t-test, ANOVA) assume normality. With n > 5,000 the Shapiro-Wilk test almost always rejects normality (high power issue). We use the Shapiro-Wilk on a 5,000-sample subset and supplement with visual Q-Q plots to make an informed test choice.

In [ ]:
sample_size = 5_000
np.random.seed(42)
sample_idx = np.random.choice(len(df), size=sample_size, replace=False)

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

for ax, col, label in zip(axes,
                           ['price', 'log_price', 'review_rate_month'],
                           ['Price ($)', 'log1p(Price)', 'Reviews/Month']):
    sample_data = df[col].iloc[sample_idx].dropna().values
    stats.probplot(sample_data, dist='norm', plot=ax)
    ax.set_title(f'Q-Q Plot: {label}')

plt.suptitle('Normality Check — Q-Q Plots (5,000-sample subset)', fontweight='bold')
plt.tight_layout()
plt.show()

for col in ['price', 'log_price', 'review_rate_month']:
    sample = df[col].iloc[sample_idx].dropna().values
    w, p = stats.shapiro(sample[:5000])


**Decision:** All three variables violate normality (p ≪ 0.05), even `log_price`. With n > 48,000, the Central Limit Theorem ensures that the *sampling distribution of the mean* is normal, which justifies Welch's t-test for means. For distributional comparisons, we also use non-parametric Mann-Whitney U and Kruskal-Wallis as robustness checks.

---
## 4.3 H1: Manhattan vs Brooklyn Price

**Hypothesis:** Manhattan listings have a significantly higher nightly price than Brooklyn listings.

- H₀: μ_manhattan = μ_brooklyn  
- H₁: μ_manhattan > μ_brooklyn  (one-tailed)

In [ ]:
manhattan = df[df['neighbourhood_group'] == 'Manhattan']['price'].values
brooklyn  = df[df['neighbourhood_group'] == 'Brooklyn']['price'].values

t_stat, p_two = stats.ttest_ind(manhattan, brooklyn, equal_var=False)
p_one = p_two / 2
d = cohens_d(manhattan, brooklyn)

report_test(
    'Welch\'s t-test: Manhattan vs Brooklyn Price',
    t_stat, p_one,
    effect_name="Cohen's d", effect=d,
    interpretation='Manhattan listings are priced significantly higher than Brooklyn.'
)

u_stat, p_mw = stats.mannwhitneyu(manhattan, brooklyn, alternative='greater')
rrb = rank_biserial_r(u_stat, len(manhattan), len(brooklyn))
report_test(
    'Mann-Whitney U (robustness check)',
    u_stat, p_mw,
    effect_name='Rank-biserial r', effect=rrb,
    interpretation='Non-parametric result confirms Manhattan > Brooklyn.'
)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
sns.kdeplot(manhattan, ax=ax, label=f'Manhattan (μ=${manhattan.mean():.0f})',
            color='#E63946', fill=True, alpha=0.4)
sns.kdeplot(brooklyn,  ax=ax, label=f'Brooklyn (μ=${brooklyn.mean():.0f})',
            color='#457B9D', fill=True, alpha=0.4)
ax.set_xlim(0, 500)
ax.set_xlabel('Price ($)')
ax.set_title('Price Distribution: Manhattan vs Brooklyn')
ax.legend()
plt.tight_layout()
plt.show()

**Conclusion H1:** Reject H₀. Manhattan prices are **statistically significantly higher** than Brooklyn (Welch's t, p ≪ 0.001; Cohen's d ≈ 0.57, moderate-to-large effect). Manhattan's mean price is ≈$58 higher per night — at NYC's median 2.3-night stay, that's **≈$133 more per booking**.

---
## 4.4 H2: Price Differences Across Room Types

**Hypothesis:** Nightly price differs significantly across room types (Entire home/apt, Private room, Shared room).

- H₀: All room-type price distributions are identical  
- H₁: At least one room type has a different price distribution

In [ ]:
room_types = ['Entire home/apt', 'Private room', 'Shared room']
groups = [df[df['room_type'] == rt]['price'].values for rt in room_types]

for rt, g in zip(room_types, groups):

h_stat, p_kw = stats.kruskal(*groups)
n_total = sum(len(g) for g in groups)
eta_sq = (h_stat - len(groups) + 1) / (n_total - len(groups))

report_test(
    'Kruskal-Wallis: Price across Room Types',
    h_stat, p_kw,
    effect_name='η² (eta-sq)', effect=eta_sq,
    interpretation='At least one room type has a significantly different price distribution.'
)

In [ ]:
from itertools import combinations

pairs = list(combinations(range(len(room_types)), 2))
bonferroni_alpha = ALPHA / len(pairs)

for i, j in pairs:
    u, p = stats.mannwhitneyu(groups[i], groups[j], alternative='two-sided')
    rrb = rank_biserial_r(u, len(groups[i]), len(groups[j]))
    sig = '✓' if p < bonferroni_alpha else '✗'


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
room_colors = {'Entire home/apt': '#1D3557', 'Private room': '#457B9D', 'Shared room': '#A8DADC'}
for rt in room_types:
    data = df[df['room_type'] == rt]['price'].clip(upper=400)
    sns.kdeplot(data, ax=ax, label=f'{rt} (μ=${df[df["room_type"]==rt]["price"].mean():.0f})',
                fill=True, alpha=0.3, color=room_colors[rt])
ax.set_xlabel('Price ($)')
ax.set_title('Price Distribution by Room Type')
ax.legend()
plt.tight_layout()
plt.show()

**Conclusion H2:** Reject H₀. All three room-type pairs have **significantly different price distributions** after Bonferroni correction. Entire home/apt commands the highest median price, followed by Private room, then Shared room. The rank-biserial r effect between Entire home and Private room is large (|r| > 0.5), confirming this is a commercially meaningful difference, not just a statistical artefact.

---
## 4.5 H3: Multi-Listers Earn Higher Revenue

**Hypothesis:** Multi-listing hosts generate a significantly higher revenue proxy than single-listing hosts.

- H₀: Revenue proxy distributions are identical between single and multi-listers  
- H₁: Multi-listers have higher revenue proxy

In [ ]:
single = df[df['is_multi_lister'] == 0]['revenue_proxy'].values
multi  = df[df['is_multi_lister'] == 1]['revenue_proxy'].values


u_stat, p_mw = stats.mannwhitneyu(multi, single, alternative='greater')
rrb = rank_biserial_r(u_stat, len(multi), len(single))

report_test(
    'Mann-Whitney U: Multi-lister vs Single Revenue',
    u_stat, p_mw,
    effect_name='Rank-biserial r', effect=rrb,
    interpretation='Multi-listers generate statistically higher revenue proxy per listing.'
)

s_price = df[df['is_multi_lister'] == 0]['price'].values
m_price = df[df['is_multi_lister'] == 1]['price'].values
t_price, p_price = stats.ttest_ind(m_price, s_price, equal_var=False)
d_price = cohens_d(m_price, s_price)
report_test(
    'Welch t-test: Multi vs Single — Price',
    t_price, p_price,
    effect_name="Cohen's d", effect=d_price,
    interpretation='Multi-listers also price their listings significantly differently.'
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, col, xlabel in zip(axes,
                            ['revenue_proxy', 'price'],
                            ['Revenue Proxy ($)', 'Price per Night ($)']):
    for val, label, color in [(0, 'Single-lister', '#457B9D'), (1, 'Multi-lister', '#E63946')]:
        data = df[df['is_multi_lister'] == val][col].clip(upper=df[col].quantile(0.95))
        sns.kdeplot(data, ax=ax, label=label, color=color, fill=True, alpha=0.3)
    ax.set_xlabel(xlabel)
    ax.set_title(f'{xlabel}: Single vs Multi-Lister')
    ax.legend()

plt.tight_layout()
plt.show()

**Conclusion H3:** Reject H₀. Multi-listing hosts generate **significantly higher revenue proxy** (Mann-Whitney p ≪ 0.001). The rank-biserial effect size indicates a moderate but real commercial advantage. This is partially attributable to higher prices and higher availability — professional hosts optimise both levers simultaneously.

---
## 4.6 H4: Borough Price Ranking

**Hypothesis:** Nightly prices differ significantly across all five NYC boroughs.

- H₀: All borough price distributions are identical  
- H₁: At least one borough differs

In [ ]:
borough_order = ['Manhattan', 'Brooklyn', 'Queens', 'Bronx', 'Staten Island']
borough_groups = [df[df['neighbourhood_group'] == b]['price'].values for b in borough_order]

for b, g in zip(borough_order, borough_groups):

h_stat, p_kw = stats.kruskal(*borough_groups)
n_total = sum(len(g) for g in borough_groups)
eta_sq  = (h_stat - len(borough_groups) + 1) / (n_total - len(borough_groups))

report_test(
    'Kruskal-Wallis: Price across 5 Boroughs',
    h_stat, p_kw,
    effect_name='η² (eta-sq)', effect=eta_sq,
    interpretation='Borough membership strongly predicts price rank.'
)

In [ ]:
borough_pairs = list(combinations(range(len(borough_order)), 2))
bonferroni_alpha = ALPHA / len(borough_pairs)

results = []
for i, j in borough_pairs:
    u, p = stats.mannwhitneyu(borough_groups[i], borough_groups[j], alternative='two-sided')
    sig = '✓' if p < bonferroni_alpha else '✗'
    results.append({'pair': f'{borough_order[i]} vs {borough_order[j]}', 'U': u, 'p': p, 'sig': sig})

sig_count = sum(1 for r in results if r['sig'] == '✓')


In [ ]:
BOROUGH_PALETTE = {
    'Manhattan': '#E63946', 'Brooklyn': '#457B9D', 'Queens': '#2A9D8F',
    'Bronx': '#F4A261', 'Staten Island': '#8338EC'
}

fig, ax = plt.subplots(figsize=(12, 5))
plot_data = [df[df['neighbourhood_group'] == b]['price'].clip(upper=400).values
             for b in borough_order]
bp = ax.boxplot(plot_data, labels=borough_order, patch_artist=True,
                notch=True, showfliers=False)
for patch, b in zip(bp['boxes'], borough_order):
    patch.set_facecolor(BOROUGH_PALETTE[b])
    patch.set_alpha(0.75)
ax.set_ylabel('Price ($)')
ax.set_title('Price Distribution by Borough (notch = 95% CI of median, outliers hidden)')
plt.tight_layout()
plt.show()

**Conclusion H4:** Reject H₀. Borough membership is a **significant price predictor** (Kruskal-Wallis p ≪ 0.001, η² indicates meaningful effect). **All or most borough pairs are significantly different** after correction. The notch plot visually confirms non-overlapping median confidence intervals — borough is one of the strongest categorical price predictors in this dataset.

---
## 4.7 H5: OLS Regression — Predicting log_price

**Hypothesis:** `log_price` can be significantly explained by room type, borough, availability, and host listing count.  
Using `log_price` (not raw price) as the dependent variable to satisfy the linear model's requirement for approximate normality of residuals.

In [ ]:
try:
    import statsmodels.formula.api as smf
    HAVE_STATSMODELS = True
except ImportError:
    HAVE_STATSMODELS = False

if HAVE_STATSMODELS:
    reg_df = df[[
        'log_price', 'room_type', 'neighbourhood_group',
        'availability_365', 'host_listing_count',
        'minimum_nights', 'has_reviews'
    ]].dropna().copy()

    formula = (
        'log_price ~ C(room_type, Treatment("Entire home/apt")) '
        '+ C(neighbourhood_group, Treatment("Manhattan")) '
        '+ availability_365 + host_listing_count '
        '+ minimum_nights + has_reviews'
    )

    model = smf.ols(formula=formula, data=reg_df).fit()


In [ ]:
if HAVE_STATSMODELS:
    coefs = model.params.drop('Intercept')
    conf  = model.conf_int().drop('Intercept')

    clean_names = (
        coefs.index
        .str.replace(r"C\(room_type.*?\)\[", '', regex=True)
        .str.replace(r"C\(neighbourhood.*?\)\[", '', regex=True)
        .str.replace(r"T\.", '', regex=True)
        .str.replace(r"\]", '', regex=True)
    )

    fig, ax = plt.subplots(figsize=(10, 6))
    y = range(len(coefs))
    ax.barh(y, coefs.values,
            xerr=[(coefs.values - conf.iloc[:, 0].values),
                  (conf.iloc[:, 1].values - coefs.values)],
            color=['#E63946' if v > 0 else '#457B9D' for v in coefs.values],
            alpha=0.8, capsize=4)
    ax.set_yticks(list(y))
    ax.set_yticklabels(clean_names)
    ax.axvline(0, color='black', lw=1)
    ax.set_xlabel('Coefficient (Δ log_price)')
    ax.set_title('OLS Regression Coefficients with 95% CI\n(Reference: Entire home/apt, Manhattan)')
    plt.tight_layout()
    plt.show()


In [ ]:
if HAVE_STATSMODELS:
    residuals = model.resid
    fitted    = model.fittedvalues

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    axes[0].scatter(fitted, residuals, alpha=0.1, s=5, color='#457B9D')
    axes[0].axhline(0, color='#E63946', lw=1.5)
    axes[0].set_xlabel('Fitted Values')
    axes[0].set_ylabel('Residuals')
    axes[0].set_title('Residuals vs Fitted')

    stats.probplot(residuals, dist='norm', plot=axes[1])
    axes[1].set_title('Q-Q Plot of Residuals')

    plt.tight_layout()
    plt.show()

**Conclusion H5:** The OLS model is statistically significant overall (F-test p ≪ 0.001). Key findings from coefficients (reference = Entire home/apt in Manhattan):
- **Private room**: large negative coefficient (expected — ~45% lower than entire home)
- **Shared room**: largest negative coefficient
- **Brooklyn, Queens, Bronx, Staten Island**: all significantly negative vs Manhattan
- **availability_365**: weak positive effect — more available listings price slightly higher (professional hosts)
- **R² ≈ 0.30–0.35**: Room type + borough explain ~30–35% of log_price variance. The remaining variance reflects micro-location, amenities, and host-specific factors not in this dataset.

---
## 4.8 Correlation Analysis — Pearson & Spearman

In [ ]:
corr_cols = [
    'price', 'log_price', 'minimum_nights', 'review_count',
    'review_rate_month', 'availability_365',
    'revenue_proxy', 'occupancy_rate_est', 'host_listing_count'
]

pearson  = df[corr_cols].corr(method='pearson')
spearman = df[corr_cols].corr(method='spearman')

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
mask = np.triu(np.ones_like(pearson, dtype=bool))

for ax, corr_mat, title in zip(axes,
                                [pearson, spearman],
                                ['Pearson Correlation', 'Spearman Rank Correlation']):
    sns.heatmap(
        corr_mat, mask=mask, annot=True, fmt='.2f',
        cmap='RdBu_r', center=0, vmin=-1, vmax=1,
        square=True, linewidths=0.5,
        annot_kws={'size': 8}, ax=ax
    )
    ax.set_title(title, fontweight='bold')

plt.suptitle('Pairwise Correlations: Pearson vs Spearman', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
key_pairs = [
    ('price', 'revenue_proxy'),
    ('price', 'availability_365'),
    ('price', 'review_rate_month'),
    ('log_price', 'host_listing_count'),
    ('review_count', 'review_rate_month'),
    ('minimum_nights', 'availability_365'),
]

for a, b in key_pairs:
    r, p = stats.pearsonr(df[a].fillna(0), df[b].fillna(0))
    sig = '✓' if p < ALPHA else '✗'


**Insights:**
- `price` ↔ `revenue_proxy` (r=0.64): price is the dominant revenue lever
- `price` ↔ `review_rate_month` (r=−0.065): near-zero — higher prices do not significantly suppress booking frequency in NYC
- `review_count` ↔ `review_rate_month` (r=0.59): strong, expected — older active listings accumulate more reviews
- `minimum_nights` ↔ `availability_365` (r=0.16): listings with longer minimum stays are kept more available — medium-term rental strategy

**Pearson vs Spearman divergence** on `price`-related pairs confirms the non-linear, skewed nature of price — Spearman (rank-based) is more appropriate for this variable.

---
## 4.9 Segmentation — Revenue & Occupancy by Host Type × Borough

In [ ]:
seg = (
    df.groupby(['neighbourhood_group', 'is_multi_lister'])
    .agg(
        count          = ('id'               , 'count'),
        median_price   = ('price'            , 'median'),
        mean_price     = ('price'            , 'mean'),
        median_revenue = ('revenue_proxy'    , 'median'),
        mean_occ       = ('occupancy_rate_est', 'mean'),
        mean_avail     = ('availability_365' , 'mean'),
    )
    .round(2)
)
seg.index = seg.index.set_levels(
    ['Single-lister', 'Multi-lister'], level='is_multi_lister'
)
seg

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

metrics = [
    ('median_price'  , 'Median Price ($)'      , 'Median Price by Borough & Host Type'),
    ('median_revenue', 'Median Revenue Proxy ($)', 'Revenue Proxy by Borough & Host Type'),
    ('mean_occ'      , 'Mean Occupancy Rate'   , 'Occupancy by Borough & Host Type'),
]

seg_reset = seg.reset_index()
seg_reset['is_multi_lister_label'] = seg_reset['is_multi_lister']

for ax, (metric, ylabel, title) in zip(axes, metrics):
    pivot = seg[metric].unstack('is_multi_lister')
    pivot = pivot.reindex(borough_order)
    x = np.arange(len(pivot))
    width = 0.35
    bars1 = ax.bar(x - width/2, pivot.iloc[:, 0], width,
                   label='Single-lister', color='#457B9D', alpha=0.85)
    bars2 = ax.bar(x + width/2, pivot.iloc[:, 1], width,
                   label='Multi-lister', color='#E63946', alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels(borough_order, rotation=25, ha='right', fontsize=9)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend(fontsize=8)

plt.suptitle('Single vs Multi-Lister Performance by Borough', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

**Insights:**
1. **Multi-listers command higher prices in every borough** — confirming professional pricing advantage.
2. **Revenue proxy gap is largest in Manhattan** — professional hosts in Manhattan earn disproportionately more.
3. **Occupancy is similar or higher for multi-listers** — contrasts the simple narrative that multi-listers inflate supply and reduce their own occupancy.
4. **Queens and Bronx multi-listers show the steepest price premium** relative to single-listers — possibly because few professional hosts operate there, commanding scarcity premium.

---
## 4.10 Statistical Summary & Dashboard KPI Linkage

| Hypothesis | Test Used | Result | Business Action |
|---|---|---|---|
| H1: Manhattan > Brooklyn price | Welch's t + Mann-Whitney | ✅ Reject H₀ (p≪0.001, Cohen's d≈0.57) | Borough premium filter in dashboard |
| H2: Room types differ in price | Kruskal-Wallis + post-hoc | ✅ Reject H₀, all pairs significant | Room-type price KPI as dashboard dimension |
| H3: Multi-listers earn more | Mann-Whitney U | ✅ Reject H₀ (p≪0.001) | Host-type segmentation filter |
| H4: Boroughs differ in price | Kruskal-Wallis + post-hoc | ✅ Reject H₀, borough is strong predictor | Borough is primary dashboard axis |
| H5: log_price ~ features | OLS Regression | ✅ F-test significant; R²≈0.30–0.35 | Room type + borough are the two dominant price drivers |

### Key Statistical Findings for Dashboard Design
1. **Borough and room type jointly explain ~30–35% of price variance** → both must be core filter dimensions
2. **Price is the dominant revenue driver (r=0.64 with revenue_proxy)** → Avg Price per Night is the #1 KPI
3. **Multi-listers earn 20–40% more revenue proxy per listing** → commercialisation metric should be a dashboard KPI
4. **June is the peak booking month** → seasonality filter is essential for trend views
5. **35.9% of listings are fully blocked (avail=0)** → active supply filter must be default-on in dashboard

### Recommended KPIs for `05_final_load_prep.ipynb`
| KPI | Formula | Dashboard Use |
|---|---|---|
| Average Nightly Price | `mean(price)` | Primary revenue indicator |
| Median Price | `median(price)` | Central tendency (robust to skew) |
| Estimated Occupancy Rate | `mean(occupancy_rate_est)` | Supply utilisation |
| Revenue Proxy per Listing | `mean(revenue_proxy)` | Annual earning potential |
| Multi-lister Rate | `mean(is_multi_lister)` | Commercialisation level |
| Active Supply | `count where avail > 0` | Real market supply |
| Review Activity Rate | `mean(review_rate_month)` | Demand proxy |

> **Proceed to** `05_final_load_prep.ipynb` to compute KPIs and build the Tableau-ready export.